### Comparação de Modelos

Este notebook tem por objetivo comparar o desempenho de diferentes modelos de machine learning, especificamente `LogisticRegression`, 
`RandomForestClassifier` e `MLPClassifier`.

#### 1. Configuração do ambiente

In [ ]:
import logging

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

RANDOM_STATE = 7
TEST_SIZE = 0.2

#### 2. Carregamento dos dados pré-processados

In [2]:
df = pd.read_csv("../data/telco_customer_churn_preprocessed.csv")

#### 3. Engenharia de Atributos

In [23]:
df["avg_charge"] = (df.total_charges / df.tenure_months).fillna(0)
df["diff_from_avg_charge"] = df.avg_charge - df.monthly_charges

#### 4. Treinamento e avaliação dos modelos

In [ ]:
numeric_features = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "cltv",
    "avg_charge",
    "diff_from_avg_charge",
]
categorical_features = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
]

X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

numeric_transformer = make_pipeline(StandardScaler())
categorical_transformer = make_pipeline(
    OneHotEncoder(
        handle_unknown="infrequent_if_exist", sparse_output=False, drop="first"
    )
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

models = {
    "logistic_regression": LogisticRegression(random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(
        random_state=RANDOM_STATE, n_estimators=200
    ),
    "mlp": MLPClassifier(random_state=RANDOM_STATE, max_iter=1000),
}

for model_name, model in models.items():
    # Create and fit pipeline
    pipeline = make_pipeline(preprocessor, SelectKBest(k=20), model)

    logger.info(f"\n=== {model_name.upper()} ===")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    logger.info(classification_report(y_test, y_pred))

INFO: 
=== LOGISTIC_REGRESSION ===
INFO:               precision    recall  f1-score   support

           0       0.85      0.90      0.87      1035
           1       0.67      0.57      0.62       374

    accuracy                           0.81      1409
   macro avg       0.76      0.74      0.75      1409
weighted avg       0.80      0.81      0.81      1409

INFO: 
=== RANDOM_FOREST ===
INFO:               precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.52      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

INFO: 
=== MLP ===
INFO:               precision    recall  f1-score   support

           0       0.85      0.91      0.88      1035
           1       0.69      0.55      0.61       374

    accuracy                           0.81      1409
   macro avg       0.77    